# Customer Shopping Behavior Analysis
Data cleaning, feature engineering and loading into SQL Server.

## 1. Load the dataset

In [1]:
import pandas as pd                                   # library to work with tables
df = pd.read_csv('customer_shopping_behavior.csv')    # load the CSV into a DataFrame
df.head()                                             # show the first 5 rows

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


## 2. Explore the dataset structure

In [2]:
df.info()    # rows, columns, data types and non-null counts

<class 'pandas.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   str    
 3   Item Purchased          3900 non-null   str    
 4   Category                3900 non-null   str    
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   str    
 7   Size                    3900 non-null   str    
 8   Color                   3900 non-null   str    
 9   Season                  3900 non-null   str    
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   str    
 12  Shipping Type           3900 non-null   str    
 13  Discount Applied        3900 non-null   str    
 14  Promo Code Used         3900 non-null   str    
 15

## 3. Summary statistics

In [3]:
df.describe(include='all')    # summary statistics for numeric and text columns

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
count,3900.000000,3900.000000,3900,3900,3900,3900.000000,3900,3900,3900,3900,3863.000000,3900,3900,3900,3900,3900.000000,3900,3900
unique,NaN,NaN,2,25,4,NaN,50,4,25,4,NaN,2,6,2,2,NaN,6,7
top,NaN,NaN,Male,Blouse,Clothing,NaN,Montana,M,Olive,Spring,NaN,No,Free Shipping,No,No,NaN,PayPal,Every 3 Months
freq,NaN,NaN,2652,171,1737,NaN,96,1755,177,999,NaN,2847,675,2223,2223,NaN,677,584
mean,1950.500000,44.068462,NaN,NaN,NaN,59.764359,NaN,NaN,NaN,NaN,3.750065,NaN,NaN,NaN,NaN,25.351538,NaN,NaN
std,1125.977353,15.207589,NaN,NaN,NaN,23.685392,NaN,NaN,NaN,NaN,0.716983,NaN,NaN,NaN,NaN,14.447125,NaN,NaN
min,1.000000,18.000000,NaN,NaN,NaN,20.000000,NaN,NaN,NaN,NaN,2.500000,NaN,NaN,NaN,NaN,1.000000,NaN,NaN
25%,975.750000,31.000000,NaN,NaN,NaN,39.000000,NaN,NaN,NaN,NaN,3.100000,NaN,NaN,NaN,NaN,13.000000,NaN,NaN
50%,1950.500000,44.000000,NaN,NaN,NaN,60.000000,NaN,NaN,NaN,NaN,3.800000,NaN,NaN,NaN,NaN,25.000000,NaN,NaN
75%,2925.250000,57.000000,NaN,NaN,NaN,81.000000,NaN,NaN,NaN,NaN,4.400000,NaN,NaN,NaN,NaN,38.000000,NaN,NaN


## 4. Check missing values

In [4]:
df.isnull().sum()    # count missing values in each column

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

## 5. Impute missing review ratings (median per category)

In [5]:
# Fill missing ratings with the median rating of the same category
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

df.isnull().sum()    # check: no missing values left

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

## 6. Rename columns to snake_case

In [6]:
df.columns = df.columns.str.lower()                                    # lowercase all column names
df.columns = df.columns.str.replace(' ', '_')                          # replace spaces with underscores
df = df.rename(columns={'purchase_amount_(usd)': 'purchase_amount'})   # simplify this column name
df.columns                                                             # check the new names

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='str')

## 7. Feature engineering: age groups

In [7]:
labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']        # names of the 4 age groups
df['age_group'] = pd.qcut(df['age'], q=4, labels=labels)          # split ages into 4 equal-sized groups
df[['age', 'age_group']].head(10)                                 # check the result

,age,age_group
0,55,Middle-aged
1,19,Young Adult
2,50,Middle-aged
3,21,Young Adult
4,45,Middle-aged
5,46,Middle-aged
6,63,Senior
7,27,Young Adult
8,26,Young Adult
9,57,Middle-aged


## 8. Feature engineering: purchase frequency in days + drop redundant column

In [8]:
# Convert purchase frequency text into number of days
frequency_mapping = {
    'Fortnightly': 14,       # every 2 weeks
    'Weekly': 7,             # every week
    'Monthly': 30,           # every month
    'Quarterly': 90,         # every 3 months
    'Bi-Weekly': 14,         # every 2 weeks
    'Annually': 365,         # every year
    'Every 3 Months': 90     # every 3 months
}
df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)   # create the numeric column

# Check if discount_applied and promo_code_used are identical
print((df['discount_applied'] == df['promo_code_used']).all())   # True means same information

# Drop the redundant column
df = df.drop('promo_code_used', axis=1)   # remove promo_code_used
df.columns                                # check final columns

True


Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='str')

## 9. Load the data into SQL Server
The password is typed at runtime with `getpass` and is never stored in this notebook.

In [9]:
from sqlalchemy import create_engine          # creates the database connection
from urllib.parse import quote_plus           # encodes special characters in the password
from getpass import getpass                   # asks for the password without saving it

# Connection details
username = "sa"                               # SQL Server user
password = quote_plus(getpass("SQL Server password: "))   # typed at runtime, never stored
host = "127.0.0.1"                            # Docker SQL Server on this Mac
port = "1433"                                 # default SQL Server port
database = "customer_behavior"                # database created in VS Code

# Create the connection engine
engine = create_engine(f"mssql+pymssql://{username}:{password}@{host}:{port}/{database}")

# Load the DataFrame into SQL Server as table "customer"
table_name = "customer"                                      # table name
df.to_sql(table_name, engine, if_exists="replace", index=False)   # replace the table if it exists

# Read back a sample to check (SQL Server uses TOP instead of LIMIT)
pd.read_sql("SELECT TOP 5 * FROM customer;", engine)

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,purchase_frequency_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,Middle-aged,14
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,Young Adult,14
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,Middle-aged,7
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,49,PayPal,Weekly,Young Adult,7
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,31,PayPal,Annually,Middle-aged,365


## 10. Export the cleaned dataset for Power BI

In [10]:
# Export the cleaned dataset for Power BI
df.to_csv('customer_behavior_cleaned.csv', index=False)   # save without the index column